# Comment Classification: Final Coding & Analysis Dataset

Before running this notebook:
1. Ollama must be running, model pulled: `ollama pull qwen3.8:27b-q4_K_M`
2. Validation of the prompt/model happened in `04c_llm_selection_eval_new.ipynb` — this notebook uses the **same** system prompt and classifier setup
3. `comments_en.csv` (English translation, column `raw_en`) is needed for the VADER valence coding

What this notebook does:
- links forum usernames to survey `id`s (all 235 posters, validated)
- classifies all comments: `pers_exp`, `emot_exp`, `pol_opin`, `breadth`, `contr` via LLM, `valence` via VADER
- builds two final datasets: **person level** (N = 559, incl. non-commenters) and **comment level** (comments of survey participants)

## Setup

In [ ]:
# importing libraries

import pandas as pd
import numpy as np
import json
import re
import time
from pathlib import Path
from itertools import product
from tqdm import tqdm
from ollama import Client
import krippendorff
from sklearn.metrics import classification_report, cohen_kappa_score

import ssl, certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

In [ ]:
# paths & settings

DATA_COMMENTS = Path("data/comments")
OUT           = Path("data/comments/final")

# data.csv: liegt entweder in data/ oder in data/comments/ - wird automatisch gesucht
SURVEY_PATH = next((p for p in [Path("data/data.csv"), DATA_COMMENTS / "data.csv"] if p.exists()), None)

PROMPT_PATH = DATA_COMMENTS / "system_prompt.txt"
EVAL_PATH   = DATA_COMMENTS / "comments_eval.xlsx"
EN_PATH     = DATA_COMMENTS / "comments_en.csv"

MODEL      = "qwen3.8:27b-q4_K_M"
PROMPT_IDS = [2, 3, 6, 20, 27, 28, 31, 32, 34, 57, 98, 99, 104, 111, 129]   # few-shot IDs aus 04c

LLM_VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "contr"]
VARS     = LLM_VARS + ["valence"]

VM2VERSION = {"VM1": "Control", "VM2": "Like", "VM3": "Like & dislike"}

OUT.mkdir(parents=True, exist_ok=True)

# check: sind alle Dateien da?
files = [SURVEY_PATH, PROMPT_PATH, EVAL_PATH, EN_PATH,
         *[DATA_COMMENTS / f"data_posts_{vm}.csv" for vm in VM2VERSION]]
for f in files:
    print("✓" if f is not None and f.exists() else "✗ FEHLT", f)
assert SURVEY_PATH is not None, "data.csv weder in data/ noch in data/comments/ gefunden"
assert all(f.exists() for f in files), "Dateien fehlen, siehe oben"

## Loading the data

The post exports are encoded in **cp1252** (not UTF-8) and contain two different date formats (ISO with microseconds, and German `dd.mm.yyyy hh:mm`). Both are handled here.

All posts written in **July 2017** are seed posts created by us before the field phase. These usernames post exclusively in July, so the separation is unambiguous.

In [ ]:
def parse_dates(s):
    a = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S.%f", errors="coerce")
    b = pd.to_datetime(s, format="%d.%m.%Y %H:%M",       errors="coerce")
    c = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S",    errors="coerce")
    return a.fillna(b).fillna(c)

def n_tokens(text):
    # quanteda-nahe Zaehlung: Woerter + Satzzeichen als eigene Tokens
    return len(re.findall(r"[\w]+(?:['-][\w]+)*|[^\w\s]", str(text)))


# survey
survey = pd.read_csv(SURVEY_PATH)
survey["last_posted"] = pd.to_datetime(survey["last_posted_at"].str.replace(" UTC", "", regex=False),
                                       errors="coerce")

# posts aller drei websites
posts = []
for vm, version in VM2VERSION.items():
    p = pd.read_csv(DATA_COMMENTS / f"data_posts_{vm}.csv", encoding="cp1252")
    p["vm"], p["version"] = vm, version
    posts.append(p)
posts = pd.concat(posts, ignore_index=True)
posts["dt"] = parse_dates(posts["created_at"])

# exakte dubletten raus (1 fall: ralf_md, VM3, post 38)
n = len(posts)
posts = posts.drop_duplicates(subset=["vm", "user", "created_at", "raw"]).reset_index(drop=True)
print(f"Dubletten entfernt: {n - len(posts)}")

# seed posts (juli)
posts["is_seed"] = posts["dt"].dt.month == 7
seed_users = set(posts.loc[posts["is_seed"], "user"])
posts["seed_user"] = posts["user"].isin(seed_users)
assert (posts["seed_user"] & ~posts["is_seed"]).sum() == 0, "Seed-User postet ausserhalb Juli!"

print(f"Posts: {len(posts)} | Seeds: {posts['is_seed'].sum()} | echte Kommentare: {(~posts['is_seed']).sum()}")
print(f"Survey: N = {len(survey)} | davon mit post_count > 0: {(survey['post_count'] > 0).sum()}")

## Linking forum usernames to survey IDs

`data.csv` contains the numeric Discourse `id`, the post exports only contain the `user` name. There is no shared column.

The bridge is `last_posted_at` in `data.csv`: the timestamp of each user's last post, exact to the second. This works as a fingerprint.

| Step | Criterion | Matches |
|---|---|---|
| 1 | `last_posted_at` exact to the second, unique on both sides | 230 |
| 2 | `post_count` + word count (timestamps only exact to the minute) | 4 |
| 3 | timestamp ±90 s **and** identical `post_count` | 1 |

The key is **(vm, username)**, not the username alone: `anonym6`, `anonym8`, `anonym9`, `anonym11`, `anonym14` exist in several VMs and are different people.

In [ ]:
def link_users(survey, posts):
    rows = []
    for vm, version in VM2VERSION.items():
        real = posts[(posts["vm"] == vm) & ~posts["seed_user"]]
        agg = (real.groupby("user")
                   .agg(n_posts=("raw", "size"), last=("dt", "max"),
                        words_est=("raw", lambda s: sum(n_tokens(x) for x in s)))
                   .reset_index())
        agg["last_s"] = agg["last"].dt.floor("s")

        sub = survey[(survey["version"] == version) & (survey["post_count"] > 0)].copy()
        sub["lp_s"] = sub["last_posted"].dt.floor("s")
        taken, matched = set(), {}

        # step 1: exakter timestamp
        cnt = agg["last_s"].value_counts()
        lut = {t: u for t, u in zip(agg["last_s"], agg["user"]) if cnt[t] == 1}
        for _, r in sub.iterrows():
            u = lut.get(r["lp_s"])
            if u is not None and u not in taken:
                taken.add(u); matched[r["id"]] = (u, "timestamp_exact")

        for _, r in sub[~sub["id"].isin(matched)].iterrows():
            rest = agg[~agg["user"].isin(taken)]
            # step 2: post_count + woerter
            c = rest[(rest["n_posts"] == r["post_count"]) &
                     (abs(rest["words_est"] - r["words"]) <= max(3, 0.08 * r["words"]))]
            if len(c) == 1:
                u = c["user"].iloc[0]; taken.add(u); matched[r["id"]] = (u, "fingerprint_n_words"); continue
            # step 3: timestamp +-90s und post_count
            c = rest[(abs((rest["last"] - r["last_posted"]).dt.total_seconds()) <= 90) &
                     (rest["n_posts"] == r["post_count"])]
            if len(c) == 1:
                u = c["user"].iloc[0]; taken.add(u); matched[r["id"]] = (u, "timestamp_min_plus_n")
            else:
                matched[r["id"]] = (None, "UNRESOLVED")

        rows += [{"id": i, "vm": vm, "version": version, "username": u, "match_method": m}
                 for i, (u, m) in matched.items()]
    return pd.DataFrame(rows)


mapping = link_users(survey, posts)
print(mapping["match_method"].value_counts().to_dict())
assert mapping["username"].notna().sum() == (survey["post_count"] > 0).sum(), "Nicht alle Poster gematcht!"
mapping.to_csv(OUT / "mapping_id_username.csv", index=False)

In [ ]:
# validierung: stimmt die anzahl verknuepfter posts pro person exakt mit post_count ueberein?
lut = {(vm, u): i for vm, u, i in zip(mapping["vm"], mapping["username"], mapping["id"])}
posts["id"] = [lut.get((vm, u)) for vm, u in zip(posts["vm"], posts["user"])]

n_linked = posts[~posts["is_seed"] & posts["id"].notna()].groupby("id").size()
check = survey.set_index("id").loc[n_linked.index, "post_count"]
print(f"Personen verknuepft: {len(n_linked)} | Abweichungen zu post_count: {(n_linked != check).sum()}")
assert (n_linked == check).all()
print("✓ Verknuepfung validiert")

## Building the comment-level dataset

`row_id = topic_postnumber` is the same key as in 04c, so `comments_eval.xlsx` and `comments_en.csv` can be merged.

**Caveat:** `row_id` is not unique across the three websites. The topic `Netiquette (3)` exists in VM1 and VM2, so `Netiquette (3)_2` and `Netiquette (3)_3` exist twice (4 short comments: berna, dana13, Rob, Konstanze). That is why there is an additional key `uid = vm_row_id`, which is unique. Merges on `row_id` are only done for unambiguous IDs.

In [ ]:
comments = posts[~posts["is_seed"]].copy()
comments["row_id"] = comments["topic"] + "_" + comments["post_number"].astype(int).astype(str)
comments["uid"]    = comments["vm"] + "_" + comments["row_id"]
comments["text"]   = comments["raw"].fillna("")
comments["n_words"] = comments["text"].map(n_tokens)
comments["linked"]  = comments["id"].notna()
comments = comments.rename(columns={"user": "username"})

assert comments["uid"].is_unique
ambiguous_row_ids = set(comments.loc[comments["row_id"].duplicated(keep=False), "row_id"])
print("row_ids, die in mehreren VMs vorkommen:", ambiguous_row_ids)

# parent-text fuer replies (innerhalb derselben website!)
text_by_uid = dict(zip(comments["uid"], comments["text"]))
seed_text = posts[posts["is_seed"] & posts["post_number"].notna()].copy()
seed_text["uid"] = seed_text["vm"] + "_" + seed_text["topic"] + "_" + seed_text["post_number"].astype(int).astype(str)
text_by_uid.update(dict(zip(seed_text["uid"], seed_text["raw"].fillna(""))))   # replies auf seed posts

def get_parent_text(row):
    if pd.notna(row.get("reply_to_post_number")):
        return text_by_uid.get(f"{row['vm']}_{row['topic']}_{int(row['reply_to_post_number'])}")
    return None

comments["parent_text"] = comments.apply(get_parent_text, axis=1)

react_cols = [c for c in comments.columns if c.startswith(("key", "value"))]
comments = comments[["uid", "row_id", "vm", "version", "id", "username", "linked", "topic",
                     "post_number", "reply_to_post_number", "dt", "text", "parent_text",
                     "n_words"] + react_cols].reset_index(drop=True)

print(f"Kommentare: {len(comments)} | mit Survey-ID: {comments['linked'].sum()} "
      f"| von {comments['id'].nunique()} Personen | replies mit parent: {comments['parent_text'].notna().sum()}")

## LLM classification

In [ ]:
# system prompt laden (derselbe wie in 04c)

with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

In [ ]:
# sanity check: kommt der richtige Systemprompt an?
print(len(SYSTEM_PROMPT))
print("LÄNGE IST KEIN BEWEIS" in SYSTEM_PROMPT)
print(SYSTEM_PROMPT[:300])

Classifier function, identical to 04c except for `num_ctx`.

**Why `num_ctx` is raised from 8192 to 16384:** the system prompt has ~30,000 characters, which is roughly 8,000–9,000 tokens in German. With `num_ctx = 8192` it does not fit completely, and with a parent comment appended even less so. Ollama then silently truncates the input. This is the most likely cause of the 7 failed rows (`-1`) in 04c (n = 128 instead of 135). Your test call in 04c already used 16384.

In [ ]:
# classifier function (aus 04c), qwen3.8:27b als default

client = Client(timeout=300)

def classify_comment(text, parent_text=None, model=MODEL):
    """Classify a single comment on all 6 codebook variables in one call.
    Returns a dict of 6 ints, or a dict of -1s if parsing fails."""
    fallback = {"pers_exp": -1, "emot_exp": -1, "pol_opin": -1,
                "breadth": -1, "valence": -1, "contr": -1}

    if not isinstance(text, str) or text.strip() == "":
        return {"pers_exp": 0, "emot_exp": 0, "pol_opin": 0,
                "breadth": 0, "valence": 4, "contr": 0}

    user_msg = f"Comment: {text}"
    if parent_text:
        user_msg += f"\n\n(This is a reply to the following parent comment: {parent_text})"

    try:
        response = client.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ],
            format="json",
            think=False,
            keep_alive="30m",
            options={"temperature": 0, "num_ctx": 16384, "num_predict": 400}
        )
    except Exception as e:
        print(f"Fehler: {e}")
        return fallback

    out = response["message"]["content"].strip()

    match = re.search(r"\{.*\}", out, re.DOTALL)
    if not match:
        return fallback

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return fallback

    result = {}
    bounds = {"pers_exp": (0,1), "emot_exp": (0,1), "pol_opin": (0,1),
              "breadth": (0,3), "valence": (1,4), "contr": (0,1)}
    for key, (lo, hi) in bounds.items():
        val = parsed.get(key, -1)
        try:
            val = int(val)
        except (TypeError, ValueError):
            val = -1
        result[key] = val if lo <= val <= hi else -1

    return result

In [ ]:
# zeittest mit 3 kommentaren, bevor der ganze loop laeuft
t0 = time.time()
for _, r in comments.head(3).iterrows():
    print(classify_comment(r["text"], r["parent_text"]))
sec = (time.time() - t0) / 3
print(f"\n{sec:.1f} s pro Kommentar -> ca. {sec * len(comments) / 60:.0f} min fuer alle {len(comments)}")

Full classification run. Results are **saved to disk every 25 comments** and the loop can be resumed: if the kernel crashes or you interrupt it, simply run the cell again and it continues where it stopped.

In [ ]:
LLM_FILE = OUT / "llm_codes_final.csv"

if LLM_FILE.exists():
    done_ids = set(pd.read_csv(LLM_FILE)["uid"])
else:
    done_ids = set()

todo = comments[~comments["uid"].isin(done_ids)]
print(f"bereits erledigt: {len(done_ids)} | offen: {len(todo)}")

buffer = []
for i, (_, row) in enumerate(tqdm(todo.iterrows(), total=len(todo), desc=f"Classifying ({MODEL})"), 1):
    codes = classify_comment(row["text"], parent_text=row["parent_text"])
    buffer.append({"uid": row["uid"], **codes})

    if len(buffer) >= 25 or i == len(todo):          # speichern IM loop
        pd.DataFrame(buffer).to_csv(LLM_FILE, mode="a", index=False, header=not LLM_FILE.exists())
        buffer = []

In [ ]:
# fehlgeschlagene zeilen (-1) nochmals versuchen
llm = pd.read_csv(LLM_FILE).drop_duplicates("uid", keep="last")
failed = llm[(llm[VARS] == -1).any(axis=1)]["uid"]
print(f"fehlgeschlagen: {len(failed)}")

if len(failed):
    retry = []
    for _, row in tqdm(comments[comments["uid"].isin(failed)].iterrows(), total=len(failed), desc="Retry"):
        retry.append({"uid": row["uid"], **classify_comment(row["text"], parent_text=row["parent_text"])})
    llm = pd.concat([llm[~llm["uid"].isin(failed)], pd.DataFrame(retry)], ignore_index=True)
    llm.to_csv(LLM_FILE, index=False)

print("verbleibende -1 pro variable:")
print((llm[VARS] == -1).sum())

## Valence via VADER

The LLM did not reach the target for `valence` in 04c, so `valence` is coded with VADER on the English translation (`comments_en.csv`, column `raw_en`). The LLM valence is kept as `valence_llm` for comparison only.

In 04c VADER was run with fixed thresholds (compound ±.05, mix .15): κ = .165, ambivalent (3) never hit. Here the three thresholds are trained on the **15 few-shot IDs** and evaluated on the **same 135 test rows** as in 04c.

In [ ]:
comments_en = pd.read_csv(EN_PATH)

comments_en["scores"]   = comments_en["raw_en"].apply(analyzer.polarity_scores)
comments_en["compound"] = comments_en["scores"].apply(lambda s: s["compound"])
comments_en["pos"]      = comments_en["scores"].apply(lambda s: s["pos"])
comments_en["neg"]      = comments_en["scores"].apply(lambda s: s["neg"])

# abdeckung: sind alle kommentare uebersetzt?
coverage = comments["row_id"].isin(comments_en["row_id"])
print(f"uebersetzt: {coverage.sum()} von {len(comments)} Kommentaren")
if not coverage.all():
    print(f"⚠️  {(~coverage).sum()} Kommentare fehlen in comments_en.csv -> valence bleibt dort NaN")

In [ ]:
def map_valence(row, pos_t, neg_t, mix_min):
    if row["pos"] >= mix_min and row["neg"] >= mix_min:
        return 3   # ambivalent
    if row["compound"] >= pos_t:
        return 1   # positiv
    if row["compound"] <= neg_t:
        return 2   # negativ
    return 4        # indifferent


# ground truth aus 04c
gt = pd.read_excel(EVAL_PATH, engine="openpyxl")
gt["row_id"]  = gt.apply(lambda r: f"{r['topic']}_{int(r['post_number'])}", axis=1)
gt["breadth"] = gt["breadth"].clip(upper=3)
gt_scored = gt.merge(comments_en[["row_id", "compound", "pos", "neg"]], on="row_id", how="left")

train = gt_scored[gt_scored["unique_id"].isin(PROMPT_IDS)]      # n = 15
test  = gt_scored[~gt_scored["unique_id"].isin(PROMPT_IDS)]     # n = 135, wie 04c

grid = product(np.arange(0.05, 0.55, 0.05),      # pos_t
               np.arange(-0.55, -0.04, 0.05),    # neg_t
               np.arange(0.05, 0.30, 0.05))      # mix_min

best = max(grid, key=lambda g: cohen_kappa_score(train["valence"],
                                                train.apply(map_valence, axis=1, args=g)))
POS_T, NEG_T, MIX_MIN = best
print(f"trainierte Schwellen (n={len(train)}): pos={POS_T:.2f} neg={NEG_T:.2f} mix={MIX_MIN:.2f}\n")

pred_fixed   = test.apply(map_valence, axis=1, args=(0.05, -0.05, 0.15))
pred_trained = test.apply(map_valence, axis=1, args=best)

print("=== fixe Schwellen (wie 04c) ===")
print("Cohen's Kappa:", round(cohen_kappa_score(test["valence"], pred_fixed), 3))
print("\n=== trainierte Schwellen ===")
print(classification_report(test["valence"], pred_trained, zero_division=0))
print("Cohen's Kappa:", round(cohen_kappa_score(test["valence"], pred_trained), 3), "  (Ziel: >= .60)")

In [ ]:
# auf alle kommentare anwenden (nur eindeutige row_ids, s. oben)
comments_en["valence"] = comments_en.apply(map_valence, axis=1, args=best)
en_unique = comments_en[~comments_en["row_id"].isin(ambiguous_row_ids)].drop_duplicates("row_id")

vader = comments[["uid", "row_id"]].merge(
    en_unique[["row_id", "valence", "compound", "pos", "neg"]], on="row_id", how="left"
).drop(columns="row_id")

print(f"valence gesetzt: {vader['valence'].notna().sum()} von {len(vader)}")
print("ohne valence (mehrdeutige row_id, manuell codieren):")
print(comments.loc[comments["row_id"].isin(ambiguous_row_ids), ["uid", "username", "text"]].to_string())

## Merging all codes

In [ ]:
llm_renamed = llm.rename(columns={"valence": "valence_llm"})

coded = (comments
         .merge(llm_renamed[["uid"] + LLM_VARS + ["valence_llm"]], on="uid", how="left")
         .merge(vader, on="uid", how="left"))

# -1 (parse-fehler) als NaN, damit sie nicht in mittelwerte eingehen
for v in LLM_VARS + ["valence_llm"]:
    coded[v] = coded[v].replace(-1, np.nan)

# depth laut codebook: summe der drei facetten (0-3)
coded["depth"] = coded[["pers_exp", "emot_exp", "pol_opin"]].sum(axis=1, min_count=3)

coded.to_csv(OUT / "comments_classified_final.csv", index=False, encoding="utf-8-sig")
print(coded[VARS + ["depth"]].describe().round(2))

## Person-level dataset incl. non-commenters

All 559 participants stay in the dataset. But for content variables, non-commenters get `NaN`, **not `0`**: someone who wrote nothing does not have "self-disclosure depth 0", the variable is simply undefined for them. A 0 would pull the means down and create spurious effects.

This matches the hurdle logic of the paper:

| Stage | Question | N |
|---|---|---|
| 1 | Does someone comment at all? (`commented` 0/1) | 559 |
| 2 | *What* does someone write, given they comment? | 235 persons |

In [ ]:
linked = coded[coded["linked"]]

person = linked.groupby("id").agg(
    n_comments    = ("uid", "size"),
    words_total   = ("n_words", "sum"),
    depth_mean    = ("depth", "mean"),
    depth_max     = ("depth", "max"),
    pers_exp_any  = ("pers_exp", "max"),
    emot_exp_any  = ("emot_exp", "max"),
    pol_opin_any  = ("pol_opin", "max"),
    pol_opin_prop = ("pol_opin", "mean"),
    breadth_mean  = ("breadth", "mean"),
    breadth_max   = ("breadth", "max"),
    contr_any     = ("contr", "max"),
    contr_prop    = ("contr", "mean"),
    compound_mean = ("compound", "mean"),
).reset_index()

# valence ist nominal -> modus statt mittelwert
valence_mode = (linked.dropna(subset=["valence"]).groupby("id")["valence"]
                .agg(lambda s: s.mode().iloc[0]).rename("valence_mode").reset_index())
person = person.merge(valence_mode, on="id", how="left")

# auf alle 559 mergen
final = survey.merge(person, on="id", how="left").copy()
final["commented"]  = (final["post_count"] > 0).astype(int)
final["n_comments"] = final["n_comments"].fillna(0).astype(int)   # anzahl: 0 ist korrekt
# inhaltsvariablen bleiben NaN fuer nicht-kommentierende!

content_cols = [c for c in person.columns if c not in ("id", "n_comments")]
assert final.loc[final["commented"] == 0, content_cols].isna().all().all()

print(f"N = {len(final)}")
print(final.groupby(["version", "commented"]).size().unstack())

## Export

In [ ]:
# person level: 1 zeile pro teilnehmer (N = 559)
final.to_csv(OUT / "final_person_level.csv", index=False, encoding="utf-8-sig")

# comment level: 1 zeile pro kommentar von survey-teilnehmern, mit survey-variablen
survey_vars = ["id", "age", "male", "edu", "pri_con_fs", "grats_gen_fs", "grats_spec_fs",
               "pri_del_fs", "self_eff_fs", "trust_gen_fs", "trust_spec_fs"]
final_comments = coded[coded["linked"]].merge(survey[survey_vars], on="id", how="left")
final_comments.to_csv(OUT / "final_comment_level.csv", index=False, encoding="utf-8-sig")

print(f"final_person_level.csv  : {len(final)} Personen ({final['commented'].sum()} mit Kommentar)")
print(f"final_comment_level.csv : {len(final_comments)} Kommentare von {final_comments['id'].nunique()} Personen")
print(f"comments_classified_final.csv : {len(coded)} Kommentare (alle, inkl. ohne Survey)")

## Sanity checks

In [ ]:
checks = {
    "Survey N = 559":                    len(final) == 559,
    "235 Poster verknuepft":             final["commented"].sum() == 235,
    "1211 echte Kommentare":             len(comments) == 1211,
    "651 Kommentare mit Survey-ID":      comments["linked"].sum() == 651,
    "keine Seed-Posts im Datensatz":     comments["username"].isin(seed_users).sum() == 0,
    "breadth in 0-3":                    coded["breadth"].dropna().between(0, 3).all(),
    "valence in 1-4":                    coded["valence"].dropna().between(1, 4).all(),
    "depth in 0-3":                      coded["depth"].dropna().between(0, 3).all(),
    "keine -1 mehr in LLM-Variablen":    coded[LLM_VARS].isna().sum().sum() == 0,
    "valence fuer alle eindeutigen":     coded.loc[~coded["row_id"].isin(ambiguous_row_ids), "valence"].notna().all(),
}
for k, v in checks.items():
    print(f"{'✓' if v else '✗'}  {k}")